<a href="https://colab.research.google.com/github/ShubhangiDimri/Indic-Meme-Understanding-Sentiment-Analysis-IMUSA-/blob/main/notebooks/IMUSA_CLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd

IMUSA_DIR = "/content/drive/MyDrive/IMUSA"

print("IMUSA folder contents:")
print(os.listdir(IMUSA_DIR))

IMUSA folder contents:
['split_train.csv', 'split_val.csv', 'results']


In [4]:
import os
import pandas as pd

IMUSA_DIR = "/content/drive/MyDrive/IMUSA"
IMAGE_DIR = f"{IMUSA_DIR}/Training_images"

train_split = pd.read_csv(f"{IMUSA_DIR}/split_train.csv")
val_split = pd.read_csv(f"{IMUSA_DIR}/split_val.csv")

all_split_ids = pd.concat(
    [train_split["Id"], val_split["Id"]]
).tolist()

missing_images = [
    image_id
    for image_id in all_split_ids
    if not os.path.exists(os.path.join(IMAGE_DIR, image_id))
]

print("Total split IDs:", len(all_split_ids))
print("Training IDs:", len(train_split))
print("Validation IDs:", len(val_split))
print("Images in folder:", len(os.listdir(IMAGE_DIR)))
print("Missing images:", len(missing_images))

if missing_images:
    print("\nFirst missing images:")
    print(missing_images[:20])
else:
    print("\n✅ Every training/validation ID has a matching image.")

Total split IDs: 3002
Training IDs: 2402
Validation IDs: 600
Images in folder: 3002
Missing images: 10

First missing images:
['image_punjabi_1174', 'image_punjabi_1176', 'image_punjabi_1175', 'image_punjabi_1171', 'image_punjabi_1172', 'image_punjabi_1178', 'image_punjabi_1173', 'image_punjabi_1179', 'image_punjabi_1177', 'image_punjabi_1180']


In [5]:
# Inspect filenames around 1171-1180

matches = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if any(str(n) in f for n in range(1171, 1181))
])

print(matches)

['image_punjabi_1171.jpg', 'image_punjabi_1172.jpg', 'image_punjabi_1173.jpg', 'image_punjabi_1174.jpg', 'image_punjabi_1175.jpg', 'image_punjabi_1176.jpg', 'image_punjabi_1177.jpg', 'image_punjabi_1178.jpg', 'image_punjabi_1179.jpg', 'image_punjabi_1180.jpg']


In [6]:
def resolve_image_path(image_id):
    image_id = str(image_id).strip()

    # Try exactly as given
    p = os.path.join(IMAGE_DIR, image_id)
    if os.path.exists(p):
        return p

    # Try common extensions
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        p = os.path.join(IMAGE_DIR, image_id + ext)
        if os.path.exists(p):
            return p

    return None


missing_after_fix = [
    image_id
    for image_id in all_split_ids
    if resolve_image_path(image_id) is None
]

print("Missing after robust lookup:", len(missing_after_fix))

if missing_after_fix:
    print(missing_after_fix[:20])
else:
    print("✅ All 3002 IDs now resolve to an image.")

Missing after robust lookup: 0
✅ All 3002 IDs now resolve to an image.


In [7]:
!pip install -q transformers accelerate pillow scikit-learn

In [8]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch

CLIP_MODEL = "openai/clip-vit-base-patch32"

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL)

clip_model = clip_model.to("cuda")

print("CLIP loaded.")
print("Device:", next(clip_model.parameters()).device)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

CLIP loaded.
Device: cuda:0


In [10]:
sample_id = train_split.iloc[0]["Id"]
sample_path = resolve_image_path(sample_id)

print("Sample ID:", sample_id)
print("Resolved path:", sample_path)

image = Image.open(sample_path).convert("RGB")

inputs = clip_processor(
    images=image,
    return_tensors="pt"
)

pixel_values = inputs["pixel_values"].to("cuda")

with torch.no_grad():
    vision_outputs = clip_model.vision_model(
        pixel_values=pixel_values
    )

    image_features = vision_outputs.pooler_output

print("Image feature shape:", image_features.shape)

Sample ID: image_punjabi_619.jpg
Resolved path: /content/drive/MyDrive/IMUSA/Training_images/image_punjabi_619.jpg
Image feature shape: torch.Size([1, 768])


In [11]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch

# Same label mapping as MuRIL
labels = ["Motivational", "Neutral", "Offensive", "Sarcasm"]

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)


class IMUSAImageDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_id = row["Id"]
        image_path = resolve_image_path(image_id)

        if image_path is None:
            raise FileNotFoundError(
                f"Could not find image for {image_id}"
            )

        image = Image.open(image_path).convert("RGB")

        processed = self.processor(
            images=image,
            return_tensors="pt"
        )

        pixel_values = processed["pixel_values"].squeeze(0)

        label = label2id[row["Category"]]

        return {
            "pixel_values": pixel_values,
            "label": torch.tensor(label, dtype=torch.long),
            "id": image_id
        }


train_image_dataset = IMUSAImageDataset(
    train_split,
    clip_processor
)

val_image_dataset = IMUSAImageDataset(
    val_split,
    clip_processor
)

print("Train images:", len(train_image_dataset))
print("Validation images:", len(val_image_dataset))

{'Motivational': 0, 'Neutral': 1, 'Offensive': 2, 'Sarcasm': 3}
Train images: 2402
Validation images: 600


In [12]:
BATCH_SIZE = 32

train_image_loader = DataLoader(
    train_image_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_image_loader = DataLoader(
    val_image_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

batch = next(iter(train_image_loader))

print("Pixel values:", batch["pixel_values"].shape)
print("Labels:", batch["label"].shape)
print("First IDs:", batch["id"][:3])

Pixel values: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
First IDs: ['image_punjabi_2984.jpg', 'image_punjabi_2253.jpg', 'image_punjabi_2265.jpg']


In [13]:
import torch
import torch.nn as nn

class CLIPImageClassifier(nn.Module):
    def __init__(self, clip_model, num_classes=4):
        super().__init__()

        self.vision_model = clip_model.vision_model

        # Freeze CLIP vision encoder
        for param in self.vision_model.parameters():
            param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, pixel_values):
        # CLIP remains frozen
        with torch.no_grad():
            outputs = self.vision_model(
                pixel_values=pixel_values
            )

            features = outputs.pooler_output

        logits = self.classifier(features)

        return logits


image_model = CLIPImageClassifier(
    clip_model,
    num_classes=4
).to("cuda")

trainable = sum(
    p.numel()
    for p in image_model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in image_model.parameters()
)

print("Device:", next(image_model.parameters()).device)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

Device: cuda:0
Total parameters: 87,653,892
Trainable parameters: 197,892


In [14]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = np.array(labels)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_split["Category"]
)

image_class_weights = torch.tensor(
    weights,
    dtype=torch.float32
).to("cuda")

print("Class weights:")

for label, weight in zip(labels, weights):
    print(f"{label:15s}: {weight:.4f}")

Class weights:
Motivational   : 0.8628
Neutral        : 0.9992
Offensive      : 14.2976
Sarcasm        : 0.5649


In [15]:
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn as nn

# Recreate class weights if not already done
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(labels),
    y=train_split["Category"]
)

image_class_weights = torch.tensor(
    weights,
    dtype=torch.float32
).to("cuda")

loss_fn = nn.CrossEntropyLoss(
    weight=image_class_weights
)

optimizer = AdamW(
    image_model.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

EPOCHS = 8

best_macro_f1 = -1
best_image_state = None
image_history = []

for epoch in range(EPOCHS):

    # -------------------------
    # TRAIN
    # -------------------------
    image_model.train()
    total_train_loss = 0

    train_bar = tqdm(
        train_image_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS} - Training"
    )

    for batch in train_bar:

        pixel_values = batch["pixel_values"].to(
            "cuda",
            non_blocking=True
        )

        labels_batch = batch["label"].to(
            "cuda",
            non_blocking=True
        )

        optimizer.zero_grad()

        logits = image_model(pixel_values)

        loss = loss_fn(
            logits,
            labels_batch
        )

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    avg_train_loss = (
        total_train_loss / len(train_image_loader)
    )

    # -------------------------
    # VALIDATION
    # -------------------------
    image_model.eval()

    total_val_loss = 0
    all_preds = []
    all_true = []

    with torch.no_grad():

        val_bar = tqdm(
            val_image_loader,
            desc=f"Epoch {epoch+1}/{EPOCHS} - Validation"
        )

        for batch in val_bar:

            pixel_values = batch["pixel_values"].to(
                "cuda",
                non_blocking=True
            )

            labels_batch = batch["label"].to(
                "cuda",
                non_blocking=True
            )

            logits = image_model(pixel_values)

            loss = loss_fn(
                logits,
                labels_batch
            )

            total_val_loss += loss.item()

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_true.extend(
                labels_batch.cpu().numpy()
            )

    avg_val_loss = (
        total_val_loss / len(val_image_loader)
    )

    # -------------------------
    # METRICS
    # -------------------------
    accuracy = accuracy_score(
        all_true,
        all_preds
    )

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            all_true,
            all_preds,
            average="macro",
            zero_division=0
        )
    )

    weighted_f1 = (
        precision_recall_fscore_support(
            all_true,
            all_preds,
            average="weighted",
            zero_division=0
        )[2]
    )

    cm = confusion_matrix(
        all_true,
        all_preds,
        labels=list(range(len(labels)))
    )

    print("\n" + "=" * 60)
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss:      {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Accuracy:        {accuracy:.4f}")
    print(f"Macro Precision: {precision:.4f}")
    print(f"Macro Recall:    {recall:.4f}")
    print(f"Macro F1:        {macro_f1:.4f}")
    print(f"Weighted F1:     {weighted_f1:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print("=" * 60)

    image_history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

    if macro_f1 > best_macro_f1:

        best_macro_f1 = macro_f1

        best_image_state = {
            k: v.cpu().clone()
            for k, v in image_model.state_dict().items()
        }

        print(
            f"*** New best Macro-F1: "
            f"{best_macro_f1:.4f} ***"
        )

print("\nCLIP image-only training complete.")
print(
    "Best validation Macro-F1:",
    best_macro_f1
)

Epoch 1/8 - Training:   0%|          | 0/76 [00:00<?, ?it/s]

Epoch 1/8 - Validation:   0%|          | 0/19 [00:00<?, ?it/s]


Epoch 1/8
Train Loss:      1.3093
Validation Loss: 1.1693
Accuracy:        0.3150
Macro Precision: 0.4385
Macro Recall:    0.4489
Macro F1:        0.2872
Weighted F1:     0.3428
Confusion Matrix:
[[119  18  33   4]
 [ 60  27  59   4]
 [  0   1   8   1]
 [ 25  32 174  35]]
*** New best Macro-F1: 0.2872 ***


Epoch 2/8 - Training:   0%|          | 0/76 [00:00<?, ?it/s]

Epoch 2/8 - Validation:   0%|          | 0/19 [00:00<?, ?it/s]

...

Epoch 7/8 - Validation:   0%|          | 0/19 [00:00<?, ?it/s]


Epoch 7/8
Train Loss:      0.8296
Validation Loss: 1.1472
Accuracy:        0.5383
Macro Precision: 0.4641
Macro Recall:    0.4906
Macro F1:        0.4457
Weighted F1:     0.5585
Confusion Matrix:
[[117  49   2   6]
 [ 48  78   3  21]
 [  1   2   3   4]
 [ 24  84  33 125]]


Epoch 8/8 - Training:   0%|          | 0/76 [00:00<?, ?it/s]

Epoch 8/8 - Validation:   0%|          | 0/19 [00:00<?, ?it/s]


Epoch 8/8
Train Loss:      0.8040
Validation Loss: 1.2444
Accuracy:        0.6283
Macro Precision: 0.5144
Macro Recall:    0.5180
Macro F1:        0.4753
Weighted F1:     0.6093
Confusion Matrix:
[[133   7   4  30]
 [ 67  35   2  46]
 [  0   0   3   7]
 [ 29  11  20 206]]

CLIP image-only training complete.
Best validation Macro-F1: 0.4990647813171435


In [16]:
import os
import torch
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/IMUSA/results/image_clip"
os.makedirs(SAVE_DIR, exist_ok=True)

# Restore BEST model (epoch 6 in this run)
image_model.load_state_dict(best_image_state)
image_model = image_model.to("cuda")

torch.save({
    "model_state_dict": image_model.state_dict(),
    "label2id": label2id,
    "id2label": id2label,
    "best_macro_f1": best_macro_f1
}, f"{SAVE_DIR}/best_model.pt")

pd.DataFrame(image_history).to_csv(
    f"{SAVE_DIR}/training_log.csv",
    index=False
)

print("Saved CLIP model to:", SAVE_DIR)
print("Best CLIP Macro-F1:", best_macro_f1)

Saved CLIP model to: /content/drive/MyDrive/IMUSA/results/image_clip
Best CLIP Macro-F1: 0.4990647813171435


In [17]:
import torch.nn.functional as F
import numpy as np

image_model.eval()

all_probs = []
all_preds = []
all_true = []
all_ids = []

with torch.no_grad():

    for batch in val_image_loader:

        pixel_values = batch["pixel_values"].to(
            "cuda",
            non_blocking=True
        )

        labels_batch = batch["label"].to("cuda")

        logits = image_model(pixel_values)

        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels_batch.cpu().numpy())
        all_ids.extend(batch["id"])

probs_array = np.array(all_probs)

val_results = val_split.copy()

val_results["predicted_label"] = [
    id2label[i] for i in all_preds
]

for i, label in id2label.items():
    val_results[f"prob_{label}"] = probs_array[:, i]

val_results.to_csv(
    f"{SAVE_DIR}/val_predictions.csv",
    index=False
)

print("Saved:", f"{SAVE_DIR}/val_predictions.csv")
print("Rows:", len(val_results))

Saved: /content/drive/MyDrive/IMUSA/results/image_clip/val_predictions.csv
Rows: 600
